# ML Model Training Pipeline
## AI-Based Personalized Study Planner

This notebook trains a Random Forest Regressor to predict student target scores.

**Steps:**
1. Load and inspect dataset
2. Preprocess features
3. Train Random Forest model
4. Evaluate performance
5. Save model and encoders

**Run cells sequentially (top to bottom)**

## Step 1: Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully")

✅ All libraries imported successfully


## Step 2: Define Paths and Configuration

In [2]:
# Define paths
DATASET_PATH = '/home/abhishek/Projects/Personalized-Study-Planner/data/dataset.csv'
MODELS_PATH = '/home/abhishek/Projects/Personalized-Study-Planner/models'

# Create models directory if it doesn't exist
os.makedirs(MODELS_PATH, exist_ok=True)

# Model configuration
TEST_SIZE = 0.2
RANDOM_STATE = 42
MODEL_TYPE = 'random_forest'  # Options: 'random_forest', 'gradient_boosting', 'linear'

# Random Forest hyperparameters
RF_N_ESTIMATORS = 100
RF_MAX_DEPTH = 15
RF_MIN_SAMPLES_SPLIT = 10
RF_MIN_SAMPLES_LEAF = 5

print(f"📁 Dataset path: {DATASET_PATH}")
print(f"📁 Models path: {MODELS_PATH}")
print(f"⚙️  Model type: {MODEL_TYPE}")
print(f"⚙️  Test size: {TEST_SIZE}")
print(f"✅ Configuration loaded")

📁 Dataset path: /home/abhishek/Projects/Personalized-Study-Planner/data/dataset.csv
📁 Models path: /home/abhishek/Projects/Personalized-Study-Planner/models
⚙️  Model type: random_forest
⚙️  Test size: 0.2
✅ Configuration loaded


## Step 3: Load and Inspect Dataset

In [6]:
# Load dataset
print("Loading dataset...")
df = pd.read_csv(DATASET_PATH)

print(f"✅ Dataset loaded: {df.shape}")
print(f"\nDataset Info:")
print(df.info())

Loading dataset...
✅ Dataset loaded: (20000, 12)

Dataset Info:
<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   student_id           20000 non-null  str    
 1   subject              20000 non-null  str    
 2   topic                20000 non-null  str    
 3   previous_score       20000 non-null  float64
 4   study_hours          20000 non-null  float64
 5   attempts             20000 non-null  int64  
 6   difficulty           20000 non-null  str    
 7   confidence_level     20000 non-null  str    
 8   days_since_revision  20000 non-null  int64  
 9   attendance           20000 non-null  float64
 10  quiz_score           20000 non-null  float64
 11  target_score         20000 non-null  float64
dtypes: float64(5), int64(2), str(5)
memory usage: 1.8 MB
None


## Step 4: Inspect First Few Rows

In [7]:
print("First 5 rows of dataset:")
print(df.head())

print("\nDataset Statistics:")
print(df.describe())

First 5 rows of dataset:
  student_id subject          topic  previous_score  study_hours  attempts  \
0   STU01103      ML  DecisionTrees           74.48         1.48        11   
1   STU01152      OS     Scheduling           69.66         0.00         1   
2   STU01170      ML     NeuralNets           69.44         1.21        20   
3   STU01021     DSA        Sorting           52.99         2.88        15   
4   STU01431      OS     Scheduling           39.70         1.84        10   

  difficulty confidence_level  days_since_revision  attendance  quiz_score  \
0       Hard             High                   10       95.53       82.27   
1       Easy           Medium                   11       76.77       74.88   
2       Hard           Medium                   56       63.48       75.66   
3     Medium              Low                   59       70.62       57.85   
4       Easy           Medium                    3       91.38       46.88   

   target_score  
0         69.50  
1

## Step 5: Check Missing Values

In [8]:
# Check missing values
missing_values = df.isnull().sum()
print("Missing values:")
print(missing_values)

if missing_values.sum() > 0:
    print("\n⚠️  Handling missing values...")
    df.fillna(df.mean(numeric_only=True), inplace=True)
    print("✅ Missing values filled")
else:
    print("✅ No missing values found")

Missing values:
student_id             0
subject                0
topic                  0
previous_score         0
study_hours            0
attempts               0
difficulty             0
confidence_level       0
days_since_revision    0
attendance             0
quiz_score             0
target_score           0
dtype: int64
✅ No missing values found


## Step 6: Explore Categorical Features

In [9]:
# Explore categorical columns
print("Categorical columns and their unique values:\n")

print(f"Subjects: {df['subject'].unique()}")
print(f"Subject count: {df['subject'].value_counts()}\n")

print(f"Difficulties: {df['difficulty'].unique()}")
print(f"Difficulty count:\n{df['difficulty'].value_counts()}\n")

print(f"Confidence Levels: {df['confidence_level'].unique()}")
print(f"Confidence count:\n{df['confidence_level'].value_counts()}")

Categorical columns and their unique values:

Subjects: <StringArray>
['ML', 'OS', 'DSA', 'Aptitude', 'DBMS', 'Python', 'CN']
Length: 7, dtype: str
Subject count: subject
CN          2949
DBMS        2921
Python      2874
Aptitude    2856
OS          2827
DSA         2792
ML          2781
Name: count, dtype: int64

Difficulties: <StringArray>
['Hard', 'Easy', 'Medium']
Length: 3, dtype: str
Difficulty count:
difficulty
Medium    6710
Hard      6660
Easy      6630
Name: count, dtype: int64

Confidence Levels: <StringArray>
['High', 'Medium', 'Low']
Length: 3, dtype: str
Confidence count:
confidence_level
Medium    6705
Low       6655
High      6640
Name: count, dtype: int64


## Step 7: Preprocess Data - Separate Features and Target

In [17]:
print("\n" + "="*60)
print("DATA PREPROCESSING")
print("="*60)

# Separate features and target
X = df.drop(['target_score', 'student_id'], axis=1)
y = df['target_score']

print(f"\nFeatures shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeatures: {list(X.columns)}")


DATA PREPROCESSING

Features shape: (20000, 10)
Target shape: (20000,)

Features: ['subject', 'topic', 'previous_score', 'study_hours', 'attempts', 'difficulty', 'confidence_level', 'days_since_revision', 'attendance', 'quiz_score']


## Step 8: Encode Categorical Features

In [18]:
# Initialize encoders dictionary
encoders = {}

# Encode categorical columns
categorical_columns = ['subject', 'topic', 'difficulty', 'confidence_level']

print("Encoding categorical columns:\n")

for col in categorical_columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    encoders[col] = le
    print(f"  ✓ {col:20} → {len(le.classes_)} classes encoded")

print(f"\n✅ All categorical columns encoded")
print(f"\nEncoded features shape: {X.shape}")

Encoding categorical columns:

  ✓ subject              → 7 classes encoded
  ✓ topic                → 42 classes encoded
  ✓ difficulty           → 3 classes encoded
  ✓ confidence_level     → 3 classes encoded

✅ All categorical columns encoded

Encoded features shape: (20000, 10)


## Step 9: Inspect Encoded Features

In [19]:
print("First 5 rows after encoding:")
print(X.head())

print("\nFeature data types:")
print(X.dtypes)

First 5 rows after encoding:
   subject  topic  previous_score  study_hours  attempts  difficulty  \
0        4      7           74.48         1.48        11           1   
1        5     33           69.66         0.00         1           0   
2        4     20           69.44         1.21        20           1   
3        3     36           52.99         2.88        15           2   
4        5     33           39.70         1.84        10           0   

   confidence_level  days_since_revision  attendance  quiz_score  
0                 0                   10       95.53       82.27  
1                 2                   11       76.77       74.88  
2                 2                   56       63.48       75.66  
3                 1                   59       70.62       57.85  
4                 2                    3       91.38       46.88  

Feature data types:
subject                  int64
topic                    int64
previous_score         float64
study_hours           

## Step 10: Split Data into Train and Test Sets

In [20]:
# Store feature names for later use
feature_names = list(X.columns)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

print(f"\n✅ Data split completed")
print(f"\nTraining set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")
print(f"\nTarget (y) shapes:")
print(f"  y_train: {y_train.shape}")
print(f"  y_test: {y_test.shape}")
print(f"\nNumber of features: {len(feature_names)}")
print(f"Features: {feature_names}")


✅ Data split completed

Training set size: (16000, 10)
Test set size: (4000, 10)

Target (y) shapes:
  y_train: (16000,)
  y_test: (4000,)

Number of features: 10
Features: ['subject', 'topic', 'previous_score', 'study_hours', 'attempts', 'difficulty', 'confidence_level', 'days_since_revision', 'attendance', 'quiz_score']


## Step 11: Check Data Distribution

In [21]:
print("Target Score Distribution:\n")

print(f"Training set statistics:")
print(f"  Min: {y_train.min():.2f}")
print(f"  Max: {y_train.max():.2f}")
print(f"  Mean: {y_train.mean():.2f}")
print(f"  Std: {y_train.std():.2f}")

print(f"\nTest set statistics:")
print(f"  Min: {y_test.min():.2f}")
print(f"  Max: {y_test.max():.2f}")
print(f"  Mean: {y_test.mean():.2f}")
print(f"  Std: {y_test.std():.2f}")

Target Score Distribution:

Training set statistics:
  Min: 0.00
  Max: 100.00
  Mean: 45.75
  Std: 19.11

Test set statistics:
  Min: 0.00
  Max: 100.00
  Mean: 45.28
  Std: 19.00


## Step 12: Initialize the Model

In [22]:
print("\n" + "="*60)
print("MODEL TRAINING")
print("="*60)

print(f"\nInitializing {MODEL_TYPE} model...\n")

if MODEL_TYPE == 'random_forest':
    model = RandomForestRegressor(
        n_estimators=RF_N_ESTIMATORS,
        max_depth=RF_MAX_DEPTH,
        min_samples_split=RF_MIN_SAMPLES_SPLIT,
        min_samples_leaf=RF_MIN_SAMPLES_LEAF,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=1
    )
    print("Model: Random Forest Regressor")
    print(f"  - n_estimators: {RF_N_ESTIMATORS}")
    print(f"  - max_depth: {RF_MAX_DEPTH}")
    print(f"  - min_samples_split: {RF_MIN_SAMPLES_SPLIT}")
    print(f"  - min_samples_leaf: {RF_MIN_SAMPLES_LEAF}")

elif MODEL_TYPE == 'gradient_boosting':
    model = GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=5,
        random_state=RANDOM_STATE,
        verbose=1
    )
    print("Model: Gradient Boosting Regressor")

else:  # linear
    model = LinearRegression()
    print("Model: Linear Regression (Baseline)")

print(f"\n✅ Model initialized")


MODEL TRAINING

Initializing random_forest model...

Model: Random Forest Regressor
  - n_estimators: 100
  - max_depth: 15
  - min_samples_split: 10
  - min_samples_leaf: 5

✅ Model initialized


## Step 13: Train the Model

In [23]:
print("\n🔄 Training the model...")
print(f"Training set size: {X_train.shape}")
print(f"Features: {X_train.shape[1]}")

start_time = pd.Timestamp.now()

# Train the model
model.fit(X_train, y_train)

training_time = (pd.Timestamp.now() - start_time).total_seconds()

print(f"\n✅ Training completed in {training_time:.2f} seconds")
print(f"✅ Model successfully trained")


🔄 Training the model...
Training set size: (16000, 10)
Features: 10


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:    0.3s



✅ Training completed in 0.91 seconds
✅ Model successfully trained


[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    0.9s finished


## Step 14: Make Predictions

In [24]:
print("Making predictions on training and test sets...\n")

# Make predictions
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

print(f"✅ Predictions completed")
print(f"  Training predictions shape: {y_pred_train.shape}")
print(f"  Test predictions shape: {y_pred_test.shape}")

# Show sample predictions
print(f"\nSample predictions (first 5 test samples):")
for i in range(min(5, len(y_pred_test))):
    print(f"  Actual: {y_test.iloc[i]:.2f}% → Predicted: {y_pred_test[i]:.2f}%")

Making predictions on training and test sets...

✅ Predictions completed
  Training predictions shape: (16000,)
  Test predictions shape: (4000,)

Sample predictions (first 5 test samples):
  Actual: 58.69% → Predicted: 62.17%
  Actual: 22.13% → Predicted: 31.75%
  Actual: 44.06% → Predicted: 42.11%
  Actual: 22.17% → Predicted: 30.47%
  Actual: 54.58% → Predicted: 57.43%


[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.0s finished


## Step 15: Evaluate Model Performance

In [25]:
print("\n" + "="*60)
print("MODEL EVALUATION METRICS")
print("="*60)

# Calculate metrics
train_mae = mean_absolute_error(y_train, y_pred_train)
test_mae = mean_absolute_error(y_test, y_pred_test)

train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))

train_r2 = r2_score(y_train, y_pred_train)
test_r2 = r2_score(y_test, y_pred_test)

print("\n📊 TRAINING SET METRICS:")
print(f"  MAE  (Mean Absolute Error):  {train_mae:.4f}")
print(f"  RMSE (Root Mean Squared Error): {train_rmse:.4f}")
print(f"  R²   (Coefficient of Determination): {train_r2:.4f}")

print("\n📊 TEST SET METRICS:")
print(f"  MAE  (Mean Absolute Error):  {test_mae:.4f}")
print(f"  RMSE (Root Mean Squared Error): {test_rmse:.4f}")
print(f"  R²   (Coefficient of Determination): {test_r2:.4f}")

print("\n" + "="*60)

# Interpretation
print(f"\n📈 MODEL INTERPRETATION:")
print(f"  • The model explains {test_r2*100:.1f}% of variance in test data")
print(f"  • Average prediction error: ±{test_mae:.2f}%")
print(f"  • Model overfitting: {'Yes' if (train_r2 - test_r2) > 0.1 else 'No'}")
print(f"  • Model performance: {'Excellent' if test_r2 > 0.85 else 'Good' if test_r2 > 0.75 else 'Fair'}")


MODEL EVALUATION METRICS

📊 TRAINING SET METRICS:
  MAE  (Mean Absolute Error):  4.4677
  RMSE (Root Mean Squared Error): 5.6749
  R²   (Coefficient of Determination): 0.9118

📊 TEST SET METRICS:
  MAE  (Mean Absolute Error):  7.0231
  RMSE (Root Mean Squared Error): 8.7179
  R²   (Coefficient of Determination): 0.7894


📈 MODEL INTERPRETATION:
  • The model explains 78.9% of variance in test data
  • Average prediction error: ±7.02%
  • Model overfitting: Yes
  • Model performance: Good


## Step 16: Feature Importance Analysis

In [26]:
if hasattr(model, 'feature_importances_'):
    print("\n" + "="*60)
    print("FEATURE IMPORTANCE ANALYSIS")
    print("="*60)
    
    # Get feature importance
    importances = model.feature_importances_
    feature_importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importances
    }).sort_values('importance', ascending=False)
    
    print("\nTop 10 Most Important Features:")
    print(feature_importance_df.to_string(index=False))
    
    # Calculate cumulative importance
    feature_importance_df['cumulative_importance'] = feature_importance_df['importance'].cumsum()
    
    print("\n📊 Cumulative Importance:")
    for idx, row in feature_importance_df.head(5).iterrows():
        print(f"  {row['feature']:20} {row['importance']:.4f} ({row['cumulative_importance']*100:.1f}% cumulative)")
    
    # Save feature importance
    feature_importance_path = os.path.join(MODELS_PATH, 'feature_importance.pkl')
    joblib.dump(feature_importance_df, feature_importance_path)
    print(f"\n✅ Feature importance saved")
else:
    print("⚠️  Feature importance not available for this model type")


FEATURE IMPORTANCE ANALYSIS

Top 10 Most Important Features:
            feature  importance
     previous_score    0.344418
   confidence_level    0.202190
         difficulty    0.100192
days_since_revision    0.096794
         quiz_score    0.092610
        study_hours    0.086719
           attempts    0.033217
         attendance    0.026258
              topic    0.011047
            subject    0.006556

📊 Cumulative Importance:
  previous_score       0.3444 (34.4% cumulative)
  confidence_level     0.2022 (54.7% cumulative)
  difficulty           0.1002 (64.7% cumulative)
  days_since_revision  0.0968 (74.4% cumulative)
  quiz_score           0.0926 (83.6% cumulative)

✅ Feature importance saved


## Step 17: Create Metrics Summary

In [27]:
# Create metrics summary
metrics_summary = {
    'train_mae': train_mae,
    'test_mae': test_mae,
    'train_rmse': train_rmse,
    'test_rmse': test_rmse,
    'train_r2': train_r2,
    'test_r2': test_r2,
    'training_time': training_time,
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
}

print("\n📋 Metrics Summary Created:")
for key, value in metrics_summary.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")

metrics_df = pd.DataFrame([metrics_summary])
print(f"\n✅ Metrics summary ready for saving")


📋 Metrics Summary Created:
  train_mae: 4.4677
  test_mae: 7.0231
  train_rmse: 5.6749
  test_rmse: 8.7179
  train_r2: 0.9118
  test_r2: 0.7894
  training_time: 0.9138
  timestamp: 2026-05-18 11:43:43

✅ Metrics summary ready for saving


## Step 18: Save the Trained Model

In [28]:
print("\n" + "="*60)
print("SAVING MODEL AND ARTIFACTS")
print("="*60)

# Save model
model_file = os.path.join(MODELS_PATH, 'random_forest_model.pkl')
joblib.dump(model, model_file)
model_size_kb = os.path.getsize(model_file) / 1024
print(f"\n✅ Model saved: {model_file}")
print(f"   Size: {model_size_kb:.2f} KB")

# Save encoders
encoders_file = os.path.join(MODELS_PATH, 'encoders.pkl')
joblib.dump(encoders, encoders_file)
print(f"\n✅ Encoders saved: {encoders_file}")
print(f"   Categories: {list(encoders.keys())}")

# Save feature names
feature_names_file = os.path.join(MODELS_PATH, 'feature_names.pkl')
joblib.dump(feature_names, feature_names_file)
print(f"\n✅ Feature names saved: {feature_names_file}")
print(f"   Features: {feature_names}")

# Save metrics
metrics_file = os.path.join(MODELS_PATH, 'training_metrics.pkl')
joblib.dump(metrics_summary, metrics_file)
print(f"\n✅ Metrics saved: {metrics_file}")

print(f"\n" + "="*60)
print(f"All files saved successfully!")
print("="*60)


SAVING MODEL AND ARTIFACTS

✅ Model saved: /home/abhishek/Projects/Personalized-Study-Planner/models/random_forest_model.pkl
   Size: 22405.16 KB

✅ Encoders saved: /home/abhishek/Projects/Personalized-Study-Planner/models/encoders.pkl
   Categories: ['subject', 'topic', 'difficulty', 'confidence_level']

✅ Feature names saved: /home/abhishek/Projects/Personalized-Study-Planner/models/feature_names.pkl
   Features: ['subject', 'topic', 'previous_score', 'study_hours', 'attempts', 'difficulty', 'confidence_level', 'days_since_revision', 'attendance', 'quiz_score']

✅ Metrics saved: /home/abhishek/Projects/Personalized-Study-Planner/models/training_metrics.pkl

All files saved successfully!


## Step 19: Verify Saved Files

In [29]:
print("\n📁 Verifying saved files...\n")

files_to_check = [
    ('random_forest_model.pkl', 'ML Model'),
    ('encoders.pkl', 'Categorical Encoders'),
    ('feature_names.pkl', 'Feature Names'),
    ('feature_importance.pkl', 'Feature Importance'),
    ('training_metrics.pkl', 'Training Metrics')
]

for filename, description in files_to_check:
    file_path = os.path.join(MODELS_PATH, filename)
    if os.path.exists(file_path):
        size_kb = os.path.getsize(file_path) / 1024
        print(f"✅ {description:25} {filename:30} ({size_kb:8.2f} KB)")
    else:
        print(f"❌ {description:25} {filename:30} (NOT FOUND)")

print(f"\n📁 Models directory: {MODELS_PATH}")
print(f"\nContents of models directory:")
for file in os.listdir(MODELS_PATH):
    file_path = os.path.join(MODELS_PATH, file)
    size = os.path.getsize(file_path) / 1024
    print(f"  {file:40} {size:10.2f} KB")


📁 Verifying saved files...

✅ ML Model                  random_forest_model.pkl        (22405.16 KB)
✅ Categorical Encoders      encoders.pkl                   (    1.68 KB)
✅ Feature Names             feature_names.pkl              (    0.15 KB)
✅ Feature Importance        feature_importance.pkl         (    1.76 KB)
✅ Training Metrics          training_metrics.pkl           (    0.30 KB)

📁 Models directory: /home/abhishek/Projects/Personalized-Study-Planner/models

Contents of models directory:
  feature_names.pkl                              0.15 KB
  random_forest_model.pkl                    22405.16 KB
  training_metrics.pkl                           0.30 KB
  feature_importance.pkl                         1.76 KB
  encoders.pkl                                   1.68 KB


## Step 20: Test Loading Saved Model

In [30]:
print("\n" + "="*60)
print("TESTING SAVED MODEL")
print("="*60)

# Load the saved model
loaded_model = joblib.load(os.path.join(MODELS_PATH, 'random_forest_model.pkl'))
loaded_encoders = joblib.load(os.path.join(MODELS_PATH, 'encoders.pkl'))
loaded_feature_names = joblib.load(os.path.join(MODELS_PATH, 'feature_names.pkl'))

print("\n✅ Model components loaded successfully")

# Test prediction with loaded model
print("\nTesting prediction with loaded model...")

# Get first 5 test samples
test_predictions = loaded_model.predict(X_test.iloc[:5])
actual_values = y_test.iloc[:5].values

print("\nComparison of Actual vs Predicted:")
print("\n{:<10} {:<15} {:<15} {:<10}".format("Sample", "Actual", "Predicted", "Error %"))
print("-" * 50)

for i in range(5):
    error_pct = abs(actual_values[i] - test_predictions[i]) / actual_values[i] * 100
    print("{:<10} {:<15.2f} {:<15.2f} {:<10.2f}%".format(
        f"Test {i+1}",
        actual_values[i],
        test_predictions[i],
        error_pct
    ))

print("\n✅ Model predictions verified successfully!")


TESTING SAVED MODEL

✅ Model components loaded successfully

Testing prediction with loaded model...

Comparison of Actual vs Predicted:

Sample     Actual          Predicted       Error %   
--------------------------------------------------
Test 1     58.69           62.17           5.92      %
Test 2     22.13           31.75           43.49     %
Test 3     44.06           42.11           4.42      %
Test 4     22.17           30.47           37.42     %
Test 5     54.58           57.43           5.22      %

✅ Model predictions verified successfully!


[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.0s finished


## Step 21: Generate Training Report

In [31]:
print("\n" + "="*70)
print(" " * 15 + "TRAINING SUMMARY REPORT")
print("="*70)

print(f"\n📅 Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

print(f"\n📊 DATASET:")
print(f"  • Total samples: {len(df)}")
print(f"  • Training samples: {len(X_train)}")
print(f"  • Test samples: {len(X_test)}")
print(f"  • Number of features: {len(feature_names)}")
print(f"  • Features: {', '.join(feature_names[:5])}...")

print(f"\n🤖 MODEL CONFIGURATION:")
print(f"  • Model type: {MODEL_TYPE}")
print(f"  • Estimators: {RF_N_ESTIMATORS}")
print(f"  • Max depth: {RF_MAX_DEPTH}")
print(f"  • Training time: {training_time:.2f} seconds")

print(f"\n📈 PERFORMANCE METRICS:")
print(f"  • Training R²: {train_r2:.4f}")
print(f"  • Test R²: {test_r2:.4f}")
print(f"  • Training MAE: {train_mae:.4f}")
print(f"  • Test MAE: {test_mae:.4f}")
print(f"  • Test RMSE: {test_rmse:.4f}")

print(f"\n💾 SAVED ARTIFACTS:")
print(f"  ✓ Model: {os.path.basename(model_file)}")
print(f"  ✓ Encoders: encoders.pkl")
print(f"  ✓ Feature Names: feature_names.pkl")
print(f"  ✓ Feature Importance: feature_importance.pkl")
print(f"  ✓ Metrics: training_metrics.pkl")

print(f"\n✅ TRAINING COMPLETED SUCCESSFULLY!")
print("="*70)

print(f"\n🚀 Next Steps:")
print(f"  1. Use the saved model for predictions")
print(f"  2. Run the Streamlit app: streamlit run app.py")
print(f"  3. Test predictions on new data")


               TRAINING SUMMARY REPORT

📅 Timestamp: 2026-05-18 11:44:37

📊 DATASET:
  • Total samples: 20000
  • Training samples: 16000
  • Test samples: 4000
  • Number of features: 10
  • Features: subject, topic, previous_score, study_hours, attempts...

🤖 MODEL CONFIGURATION:
  • Model type: random_forest
  • Estimators: 100
  • Max depth: 15
  • Training time: 0.91 seconds

📈 PERFORMANCE METRICS:
  • Training R²: 0.9118
  • Test R²: 0.7894
  • Training MAE: 4.4677
  • Test MAE: 7.0231
  • Test RMSE: 8.7179

💾 SAVED ARTIFACTS:
  ✓ Model: random_forest_model.pkl
  ✓ Encoders: encoders.pkl
  ✓ Feature Names: feature_names.pkl
  ✓ Feature Importance: feature_importance.pkl
  ✓ Metrics: training_metrics.pkl

✅ TRAINING COMPLETED SUCCESSFULLY!

🚀 Next Steps:
  1. Use the saved model for predictions
  2. Run the Streamlit app: streamlit run app.py
  3. Test predictions on new data
